In [1]:
# change to the Fgpt directory
%cd ..

/home/ssivanes/Fgpt


/data/ssivanes/fparser-venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from processor import Processor
from extractor import Extractor
from isolator import Isolator

from fparser.two import Fortran2003 as F23
from fparser.two.utils import walk
import logging
from typing import Generator, Optional, Type, Union,Dict,List,Literal
import numpy as np
import os

processor = Processor()

INFO     Processor initialized.

In [3]:
rest_of_path = "/data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/"
target_module = "hydrol" # hydrol explicitsnow
work = os.getenv("work")

isolator = Isolator(rest_of_path, target_module, work,False)

cls = Extractor(isolator.module_dir_sp, isolator.module_tree_sp)
cls.find_subroutines()
cls.extract_loop_indices()

INFO     Processor initialized.

INFO     Successfully parsed file:                                                                                 
         /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/hydrol_org.f90

INFO     Successfully parsed string!

INFO     Processor initialized.

In [4]:
%reload_ext autoreload
%autoreload 2
from transformer import Transformer

In [5]:
transformer = Transformer("/home/ssivanes/Fgpt/benchmark",isolator,cls,None,config_path = "/home/ssivanes/Fgpt/template.yaml")
# transformer.subroutine_name = subroutine_key

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: Transformer                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
import ast
empty_ast = ast.Module(body=[], type_ignores=[])
# Let's say that the AST fortran and AST python works in a similar manner, this will allows us to do the following:
# - modify/add to the python AST structure directly without the need to work with indentation, thus during the translation we modify the AST
#   strucutre
# - useful when integrating with multiples files thus allowing us to modify between both files

# Let's start by creating a python AST structure for the code template
# Then compare these structures, to do so we will use the code template from the module_global file which is set as out_module
subroutine_key = "hydrol_split_soil"
# hydrol_diag_soil hydrol_soil_froz hydrol_soil_setup hydrol_split_soil hydrol_soil_coef hydrol_soil_tridiag hydrol_root_profile 
# hydrol_soil_infilt hydrol_soil_smooth_over_mcs2 hydrol_soil_smooth_under_mcr,hydrol_tmc_update

# hydrol_muff_radial_resolution hydrol_muff_radial_coef_setup

# explicitsnow_age explicitsnow_compactn explicitsnow_fall explicitsnow_gone explicitsnow_levels explicitsnow_melt_refrz explicitsnow_maxmass
# explicitsnow_profile explicitsnow_subli explicitsnow_transf
out_module = processor.out_module_fortran(subroutine_key)

INFO     Successfully parsed module code

In [7]:
cls.call_within_sub # THis is only for the subroutines 

defaultdict(set,
            {'hydrol_main': {'explicitsnow_main',
              'hydrol_alma',
              'hydrol_canop',
              'hydrol_flood',
              'hydrol_hydraulic_arch_tuzet_calc',
              'hydrol_nudge_mc_diag',
              'hydrol_nudge_snow',
              'hydrol_soil',
              'hydrol_vegupd'},
             'hydrol_vegupd': {'hydrol_tmc_update'},
             'hydrol_soil': {'hydrol_diag_soil',
              'hydrol_diag_soil_flux',
              'hydrol_nudge_mc',
              'hydrol_root_profile',
              'hydrol_soil_coef',
              'hydrol_soil_froz',
              'hydrol_soil_infilt',
              'hydrol_soil_setup',
              'hydrol_soil_smooth_over_mcs2',
              'hydrol_soil_smooth_under_mcr',
              'hydrol_soil_tridiag',
              'hydrol_split_soil'},
             'hydrol_hydraulic_arch_tuzet_calc': {'hydrol_hydraulic_arch_tuzet_muff',
              'hydrol_hydraulic_arch_tuzet_resist'},
      

In [8]:
subroutine_tree = cls.subroutines[subroutine_key]
print(subroutine_tree)



  !!
  !& ================================================================================================================================
  !! SUBROUTINE   : hydrol_split_soil
  !!
  !>\BRIEF        Splits 2d variables into 3d variables, per soiltile (_ns suffix), at the beginning of hydrol
  !!              At this stage, the forcing fluxes to hydrol are transformed from grid-cell averages
  !!              to mean fluxes over vegtot=sum(soiltile)
  !!
  !! DESCRIPTION  :
  !! 1. Split 2d variables into 3d variables, per soiltile
  !! 1.1 Throughfall
  !! 1.2 Bare soil evaporation
  !! 1.2.2 ae_ns new
  !! 1.3 transpiration
  !! 1.4 root sink
  !! 2. Verification: Check if the deconvolution is correct and conserves the fluxes
  !! 2.1 precisol
  !! 2.2 ae_ns and evapnu
  !! 2.3 transpiration
  !! 2.4 root sink
  !!
  !! RECENT CHANGE(S) : 2016 by A. Ducharne to match the simplification of hydrol_soil
  !!
  !! MAIN OUTPUT VARIABLE(S) :
  !!
  !! REFERENCE(S) :
  !!
  !! FLOWCHART  

In [9]:
cls.general_usage_dict

defaultdict(None, {})

In [10]:
cls.extract_intent(subroutine_key, subroutine_tree,cls.call_within_sub[subroutine_key])

In [11]:
cls.clean_subroutine(subroutine_key, subroutine_tree)

WARNING  Warning: Name tot_bare_soil is not used REAL(KIND = r_std), DIMENSION(kjpindex), INTENT(IN) ::            
         tot_bare_soil.

WARNING  Incorrect intent for us. Expected: IN, Found: INOUT. Correct it!

WARNING  Original Declaration Statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm, nslm), INTENT(INOUT) ::
         us

INFO     Modified Declaration Statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm, nslm), INTENT(IN) :: us

In [12]:
cls.find_variables(subroutine_tree, subroutine_key)

In [13]:
cls.extract_names(subroutine_key)

In [14]:
cls.find_global_variables(isolator.module_dir_sp, isolator.module_tree_sp, cls.var_global[subroutine_key], subroutine_key)

INFO     Processor initialized.

INFO     ⏳... Searching for variable 'ok_hydrol_arch'

INFO     Module 'ioipsl' is added into the queue.

INFO     Module 'xios_orchidee' is added into the queue.

INFO     Module 'constantes' is added into the queue.

...

INFO     Checking the child module ...'constantes_var'

INFO     'min_sechiba' is found in 'constantes_var' of the module 'constantes_var'

INFO     REAL(KIND = r_std), PARAMETER :: min_sechiba = 1.E-8_r_std

INFO     The containing directory is: /data/ssivanes/modipsl_truck_opt/modeles/ORCHIDEE/src_sechiba/

INFO     ✅ Variable found!

In [15]:
shape_to_search = cls.shapes_variables[subroutine_key] - cls.scalar_variables[subroutine_key] - cls.var_global[subroutine_key]

if shape_to_search:
    cls.find_global_variables(isolator.module_dir_sp, isolator.module_tree_sp, shape_to_search, subroutine_key)
    cls.var_global[subroutine_key].update(shape_to_search)

In [16]:
print(shape_to_search)

set()


In [17]:
cls.extract_array_info(cls.dec_global[subroutine_key], cls.var_dummy[subroutine_key],subroutine_key)

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: rootsink

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm) :: precisol

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: vegetmax_soil

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(nvm) :: pref_soil_veg

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tr_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: precisol_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns

In [18]:
# NOw we add the functions 
cls.dec_global[subroutine_key] # Since these functions are called inside other subroutines as declared variables 

defaultdict(list,
            {'ok_hydrol_arch': [Type_Declaration_Stmt(Intrinsic_Type_Spec('LOGICAL', None), Attr_Spec_List(',', (Attr_Spec('SAVE'),)), Entity_Decl_List(',', (Entity_Decl(Name('ok_hydrol_arch'), None, None, None),)))],
             'rootsink': [Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('ALLOCATABLE'), Attr_Spec('SAVE'), Dimension_Attr_Spec('DIMENSION', Assumed_Shape_Spec_List(',', (Assumed_Shape_Spec(None, None), Assumed_Shape_Spec(None, None), Assumed_Shape_Spec(None, None)))))), Entity_Decl_List(',', (Entity_Decl(Name('rootsink'), None, None, None),))),
              Allocate_Stmt(None, Allocation_List(',', (Allocation(Name('rootsink'), Allocate_Shape_Spec_List(',', (Allocate_Shape_Spec(None, Name('kjpindex')), Allocate_Shape_Spec(None, Name('nslm')), Allocate_Shape_Spec(None, Name('nstm'))))),)), Alloc_Opt_List(',', (Alloc_Opt('STAT', Name('ier')),)))],
             'ipslerr_p': [Use_Stm

In [19]:
cls.subroutines.keys()

dict_keys(['hydrol_main', 'hydrol_tmc_update', 'hydrol_canop', 'hydrol_vegupd', 'hydrol_flood', 'hydrol_soil', 'hydrol_soil_infilt', 'hydrol_soil_smooth_under_mcr', 'hydrol_soil_smooth_over_mcs', 'hydrol_soil_smooth_over_mcs2', 'hydrol_diag_soil_flux', 'hydrol_soil_tridiag', 'hydrol_soil_coef', 'hydrol_soil_froz', 'hydrol_soil_setup', 'hydrol_split_soil', 'hydrol_diag_soil', 'hydrol_alma', 'hydrol_nudge_mc', 'hydrol_nudge_mc_diag', 'hydrol_nudge_snow', 'hydrol_hydraulic_arch_tuzet_calc', 'hydrol_hydraulic_arch_tuzet_resist', 'hydrol_hydraulic_arch_tuzet_muff', 'hydrol_muff_radial_coef_setup', 'hydrol_muff_radial_resolution', 'hydrol_root_profile'])

# Functions

In [20]:
for key, values in cls.dec_global[subroutine_key].items():
    if walk(values, F23.Function_Subprogram):
        function_tree_org = values[1]
        function_key = values[0].tostr()
        if function_key in cls.subroutines: # THis would means that we have already isolated the function add has been added onto the 
            # call_within_sub parameters
            function_tree = cls.subroutines[function_key]
            cls.call_within_sub[subroutine_key].add(function_key)
        else:
            function_tree = isolator.isolate_child_function(cls, function_tree_org, function_key, subroutine_key)

In [21]:
# print(f"Dummy args list of the funciton : {cls.dummy_arg_list[function_key]}")
# print(f"Acutal args list of the function : {cls.actual_arg_spec_list[function_key]}")

In [22]:
print(cls.call_within_sub[subroutine_key])

set()


In [23]:
# print(cls.var_dummy[function_key])

In [24]:
# cls.var_modif_info[function_key]

In [25]:
# global_function_tree = transformer.update_global_python(function_key,cls_mode = True,for_loop=False)

In [26]:
#print(ast.unparse(global_function_tree))

In [27]:
# transformer.subroutine_name = function_key
# main_function_tree = transformer.update_main_python(global_function_tree)

In [28]:
# print(ast.unparse(ast.fix_missing_locations(main_function_tree)))

In [29]:
function_keys = None
function_keys = cls.call_within_sub[subroutine_key]
if function_keys:
    for function_key in cls.call_within_sub[subroutine_key]:
        isolator.collect_global_vars_decl(cls.dec_global[function_key], cls.dec_global[subroutine_key])

# Subroutines

In [30]:
isolator.processor.add_declarations(cls.dec_global[subroutine_key], cls.var_modif_info[subroutine_key])

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns

INFO     Successfully parsed statement:                                                                            
         IF (.NOT. ALLOCATED(ae_ns)) THEN                                                                          
           ALLOCATE(ae_ns(kjpindex, nstm), STAT = ier)                                                             
         END IF

INFO     Successfully generated allocation statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv

INFO     Successfully parsed statement:                                                                            
         IF (.NOT. ALLOCATED(humrelv)) THEN                                                                        
           ALLOCATE(humrelv(kjpindex, nvm, nstm), STAT = ier)                                                      
         END IF

...

INFO     Successfully parsed statement:                                                                            
         READ(1363, IOSTAT = ier) tr_ns                                                                            
         IF (ier /= 0) THEN                                                                                        
           WRITE(*, *) 'Error reading from file for tr_ns. ', ' IOSTAT : ', ier                                    
         END IF

INFO     Successfully parsed statement:                                                                            
         READ(1363, IOSTAT = ier) vegetmax_soil                                                                    
         IF (ier /= 0) THEN                                                                                        
           WRITE(*, *) 'Error reading from file for vegetmax_soil. ', ' IOSTAT : ', ier                            
         END IF

INFO     Successfully parsed statement:                                                                            
         READ(1363, IOSTAT = ier) vegtot                                                                           
         IF (ier /= 0) THEN                                                                                        
           WRITE(*, *) 'Error reading from file for vegtot. ', ' IOSTAT : ', ier                                   
         END IF

INFO     processing initialization completed!

INFO     Declarations and allocations processed successfully

In [31]:
print(cls.dec_global[subroutine_key].keys(), len(cls.dec_global[subroutine_key].keys()))

dict_keys(['ok_hydrol_arch', 'rootsink', 'ipslerr_p', 'dt_sechiba', 'precisol', 'is_tuzet_hydrol_arch', 'vegetmax_soil', 'pref_soil_veg', 'allowed_err', 'tr_ns', 'zero', 'humrelv', 'check_cwrr', 'vegtot', 'numout', 'kilo_to_unit', 'precisol_ns', 'ae_ns', 'min_sechiba']) 19


In [32]:
# cls.dec_global[function_key]

## Global module

In [33]:
cls.dec_global[subroutine_key]["snow3lgrain_0d"]

[]

In [34]:
for key in list(cls.dec_global[subroutine_key].keys()):
    print(cls.dec_global[subroutine_key][key])

[Type_Declaration_Stmt(Intrinsic_Type_Spec('LOGICAL', None), Attr_Spec_List(',', (Attr_Spec('SAVE'),)), Entity_Decl_List(',', (Entity_Decl(Name('ok_hydrol_arch'), None, None, None),)))]
[Type_Declaration_Stmt(Intrinsic_Type_Spec('REAL', Kind_Selector('(', Name('r_std'), ')')), Attr_Spec_List(',', (Attr_Spec('ALLOCATABLE'), Attr_Spec('SAVE'), Dimension_Attr_Spec('DIMENSION', Assumed_Shape_Spec_List(',', (Assumed_Shape_Spec(None, None), Assumed_Shape_Spec(None, None), Assumed_Shape_Spec(None, None)))))), Entity_Decl_List(',', (Entity_Decl(Name('rootsink'), None, None, None),))), Allocate_Stmt(None, Allocation_List(',', (Allocation(Name('rootsink'), Allocate_Shape_Spec_List(',', (Allocate_Shape_Spec(None, Name('kjpindex')), Allocate_Shape_Spec(None, Name('nslm')), Allocate_Shape_Spec(None, Name('nstm'))))),)), Alloc_Opt_List(',', (Alloc_Opt('STAT', Name('ier')),)))]
[Use_Stmt(None, None, Name('ioipsl_para'), ', ONLY:', Only_List(',', (Name('ipslerr_p'),)))]
[Type_Declaration_Stmt(Intrinsi

In [35]:
transformer.retreive_variable_order()
print(transformer.variable_order)

['check_cwrr', 'dt_sechiba', 'ok_hydrol_arch', 'ae_ns', 'humrelv', 'precisol', 'precisol_ns', 'pref_soil_veg', 'rootsink', 'tr_ns', 'vegetmax_soil', 'vegtot']


In [36]:
# transformer.subroutine_name = subroutine_key
transformer.cls_mode = True
transformer.global_state = True

In [37]:
code_template = transformer.out_module_python()
# print(ast.dump(code_template,indent=4))
print(ast.unparse(ast.fix_missing_locations(code_template)))

import numpy as np
import logging
from scipy.io import FortranFile
import os

class Global_module:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)

    def declaration_initialization(self):
        pass


In [38]:
def ast_walk(node, node_type):
    """
    Recursively walks an AST tree, yielding all nodes or nodes of a specific type.
    """
    if node_type is None or isinstance(node, node_type):
        yield node
    for child in ast.iter_child_nodes(node):
        yield from ast_walk(child, node_type)

In [39]:
transformer.pre_init_variables(code_template)
# print(transformer.pre_init)

In [40]:
declaration_stmts = list(cls.dec_global[subroutine_key].values())
ast_nodes = transformer.convert_SPECIFICATION_PART(declaration_stmts,False,cls_mode=True)

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: rootsink

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm) :: precisol

...

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: precisol_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

In [41]:
assign_nodes = []
procedure_nodes = []
        
for node in ast_nodes:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        procedure_nodes.append(node)
    elif isinstance(node, (ast.Assign, ast.Assign)):
        assign_nodes.append(node)

In [42]:
cls.dec_global[subroutine_key]["imax"]

[]

In [43]:
code = """
USE my_module, ONLY : func,var1,var2
"""
processor.parse_fortran_statement(code)

INFO     Successfully parsed statement:                                                                            
         USE my_module, ONLY: func, var1, var2

Specification_Part(Use_Stmt(None, None, Name('my_module'), ', ONLY:', Only_List(',', (Name('func'), Name('var1'), Name('var2')))))

In [44]:
code = """
USE my_module, ONLY : func => f,var1
"""
processor.parse_fortran_statement(code)


INFO     Successfully parsed statement:                                                                            
         USE my_module, ONLY: func => f, var1

Specification_Part(Use_Stmt(None, None, Name('my_module'), ', ONLY:', Only_List(',', (Rename(None, Name('func'), Name('f')), Name('var1')))))

In [45]:
transformer.pre_init_variables(code_template)

In [46]:
transformer.search_dependant_variables(declaration_stmts)
# transformer.dependant_variables

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: rootsink

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm) :: precisol

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: vegetmax_soil

INFO     Combined statement: INTEGER(KIND = i_std), DIMENSION(nvm) :: pref_soil_veg

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tr_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: precisol_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns

In [47]:
transformer.separate_scalar(subroutine_key)
transformer.retreive_variable_order()
print(transformer.variable_order,len(transformer.variable_order))

['check_cwrr', 'dt_sechiba', 'ok_hydrol_arch', 'ae_ns', 'humrelv', 'precisol', 'precisol_ns', 'pref_soil_veg', 'rootsink', 'tr_ns', 'vegetmax_soil', 'vegtot'] 12


In [48]:
assign_map = {}
for assign_node in assign_nodes:
    target = assign_node.targets[0]
    name = target.id if isinstance(target, ast.Name) else target.attr
    assign_map[name] = assign_node 

nodes = [assign_map[scalar] for scalar in transformer.scalar if scalar in assign_map]
read_ast = transformer.prepare_read_code_global_template(assign_nodes,subroutine_key)
# print(ast.unparse(ast.fix_missing_locations(read_ast)))

In [49]:
# read_ast = transformer.init_dependant_variables(read_ast,assign_nodes)

In [50]:
# print(ast.unparse(read_ast))

In [51]:
# transformer.for_loop = False
# transformer.cls_mode = True
# class_tree = transformer.transform_to_class(ast_nodes)

In [52]:
tree = transformer.update_global_python(subroutine_key,cls_mode = True,for_loop=True)

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nslm, nstm) :: rootsink

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm) :: precisol

...

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: tr_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nvm, nstm) :: humrelv

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex) :: vegtot

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: precisol_ns

INFO     Combined statement: REAL(KIND = r_std), DIMENSION(kjpindex, nstm) :: ae_ns

In [53]:
print(ast.unparse(tree))

import numpy as np
import logging
from scipy.io import FortranFile
import os

class Global_module_hydrol_split_soil:

    def __init__(self):
        self.nsnow = np.int32(3)
        self.nslm = np.int32(11)
        self.nvm = np.int32(15)
        self.nstm = np.int32(3)
        self.kjpindex = np.int32(4717)
        self.ier = np.int32(0)
        self.ic0 = np.int32(0)
        self.ic = np.int32(0)
        self.icr = np.float64(0.0)
        self.start_time = np.float64(0.0)
        self.stop_time = np.float64(0.0)
        self.is_tuzet_hydrol_arch = np.bool(False)
        self.allowed_err = np.float64(2e-08)
        self.zero = np.float64(0.0)
        self.numout = np.int32(6)
        self.kilo_to_unit = np.float64(1000.0)
        self.min_sechiba = np.float64(1e-08)
        self.check_cwrr = np.bool(False)
        self.dt_sechiba = np.float64(0)
        self.ok_hydrol_arch = np.bool(False)
        self.ae_ns = np.zeros((self.kjpindex, self.nstm), dtype=np.float64)
        self.humrel

In [54]:
transformer.transfer_to_pyfile(tree,subroutine_key=subroutine_key,folder_name=target_module)

## Main module

In [55]:
cls.call_subroutines["snow3lhold_1d"]

[]

In [56]:
print(cls.dummy_arg_list['snow3lhold_1d'])

[]


In [57]:
templates = transformer.load_code_templates(transformer.config_path)
out_main = templates["Python_templates"]["Python_main_template"]["template"]
out_main_template = ast.parse(out_main)
print(ast.unparse(out_main_template))

import os
import time
import functools
import numpy as np
import logging
from scipy.io import FortranFile

def main():
    print('--- inside the main program ---')
if __name__ == '__main__':
    main()


In [58]:
# cls_info, import_nodes, instance_nodes = transformer.create_cls_info(tree,subroutine_key)

In [59]:
# transformer.subroutine_name = subroutine_key
main_tree = transformer.update_main_python(tree,subroutine_key)

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

...

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

INFO     Successfully removed INTENT and SAVE attributes from statements

In [60]:
print(ast.unparse(main_tree))

import os
import time
import functools
import numpy as np
import logging
from scipy.io import FortranFile
from module_global import Global_module_hydrol_split_soil

def read_dummy(e_frac, evap_bare_lim, evap_bare_lim_ns, F_absorption, humrel, soiltile, tot_bare_soil, transpir, us, veget_max, vevapnu):
    print(f'--- inside the read dummy routine for hydrol_split_soil ---')
    path = f'/home/ssivanes/Fgpt/benchmark/hydrol_split_soil/dummy.bin'
    ffile = FortranFile(path, 'r')
    for var in [e_frac, evap_bare_lim, evap_bare_lim_ns, F_absorption, humrel, soiltile, tot_bare_soil, transpir, us, veget_max, vevapnu]:
        if isinstance(var, np.ndarray):
            arr_shape = var.shape
            if var.dtype == np.float64:
                data = ffile.read_reals(np.float64)
            elif var.dtype == np.int32 or var.dtype == np.bool:
                data = ffile.read_ints(np.int32)
            if data.size != np.prod(arr_shape):
                continue
            var[:] = data

In [61]:
transformer.transfer_to_pyfile(main_tree,subroutine_key,folder_name=target_module,python_file_type="main")

In [62]:
# Testing all the transformed python files:
# transformer.compile_and_run(os.getcwd(),"hydrol")

In [63]:
for elements in cls.all_array_info.values():
    print(elements.keys())

dict_keys(['rootsink', 'precisol', 'vegetmax_soil', 'pref_soil_veg', 'tr_ns', 'humrelv', 'vegtot', 'precisol_ns', 'ae_ns', 'e_frac', 'evap_bare_lim', 'evap_bare_lim_ns', 'F_absorption', 'humrel', 'soiltile', 'tot_bare_soil', 'transpir', 'us', 'veget_max', 'vevapnu', 'tmp_check1', 'tmp_check2', 'tmp_check3'])


In [64]:
cls.loop_dict

defaultdict(set,
            {'nslm': {'isl', 'jsl'},
             'kjpindex': {'ipts', 'ji'},
             'nvm': {'ivm', 'jv'},
             'nstm': {'ist', 'jst'},
             'itopmax': {'jsl'},
             'nslm - 1': {'jsl'},
             'imax - 1': {'ii'},
             'imin': {'ii'},
             '4': {'jsl'},
             'nslm - 2': {'jsl'},
             '2': {'jsl'},
             '1': {'jrp', 'jsl'},
             'nbp_glo': {'ji'},
             'nsnow': {'jg'},
             'nrp': {'jrp'},
             'nrp - 1': {'jrp'}})

# F2NP Test

In [65]:
from f2np import F2NP
f2np = F2NP(cls)

do_stat = F23.Nonlabel_Do_Stmt(" DO jjj = locflag(jj, 1), locflag(jj, 2) - 1")


╭───────────────────────────────────────── Fortran → Python Transformer ──────────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [66]:
do_stat.children[-1].children[1][1]

[Part_Ref(Name('locflag'), Section_Subscript_List(',', (Name('jj'), Int_Literal_Constant('1', None)))),
 Level_2_Expr(Part_Ref(Name('locflag'), Section_Subscript_List(',', (Name('jj'), Int_Literal_Constant('2', None)))), '-', Int_Literal_Constant('1', None))]

In [67]:
_,_,module_stack = f2np.recursive_ast(subroutine_tree)

In [68]:
print(ast.unparse(ast.fix_missing_locations(module_stack[0])))
# print(ast.dump(module_stack[0],indent=4))

def hydrol_split_soil(kjpindex, veget_max, soiltile, vevapnu, transpir, humrel, evap_bare_lim, evap_bare_lim_ns, tot_bare_soil, us, e_frac, F_absorption):
    tmp_check1 = np.zeros((kjpindex,), dtype=np.float64)
    tmp_check2 = np.zeros((kjpindex,), dtype=np.float64)
    tmp_check3 = np.zeros((kjpindex, nstm), dtype=np.float64)
    precisol_ns[:, :] = zero
    for jv in range(0, nvm, 1):
        for ji in range(0, kjpindex, 1):
            jst = pref_soil_veg[jv]
            if veget_max[ji, jv] > min_sechiba and soiltile[ji, jst] * vegtot[ji] > min_sechiba:
                precisol_ns[ji, jst] = precisol_ns[ji, jst] + precisol[ji, jv] / (soiltile[ji, jst] * vegtot[ji])
    ae_ns[:, :] = zero
    for jst in range(0, nstm, 1):
        for ji in range(0, kjpindex, 1):
            if evap_bare_lim[ji] > min_sechiba:
                ae_ns[ji, jst] = vevapnu[ji] * evap_bare_lim_ns[ji, jst] / evap_bare_lim[ji]
    tr_ns[:, :] = zero
    for jv in range(0, nvm, 1):
        jst = pref_soil_ve

In [69]:
cls.call_within_sub['explicitsnow_fall']

set()